# GRU4Rec Prototype — Session-Based Next-Item Prediction

**Purpose of this notebook:** build and train a small, working version of the model on a *tiny slice* of data (one day, not the full 20M rows) so we can:

1. Verify the whole pipeline (sessions -> vocab -> pairs -> model -> training -> evaluation) actually works end to end.
2. Get a **real, measured** CPU epoch time on this machine, instead of a theoretical estimate.
3. Learn every concept the model uses, and *why* it's built this way, before scaling up.

Once this works, the same logic gets promoted into `src/model/*.py` and pointed at the full 20M-row export — nothing conceptually changes, only the data size and a few constants.

Notebooks are for exploration only per the project layout — this one is intentionally verbose with explanations. The clean, reusable version of this logic belongs in `src/`, not here.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import pad_sequences


def find_project_root(marker="requirements.txt"):
    """Walk upward from the notebook's working directory until we find
    the repo root. Jupyter's cwd depends on where it was launched from,
    so we don't hardcode a relative '../' path."""
    path = Path.cwd()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    BATCH_SIZE, DATA_INTERIM_DIR, EMBED_DIM, GRU_UNITS, LEARNING_RATE,
    MAX_SEQ_LEN, NUM_SAMPLED_NEGATIVES, NUM_SPECIAL_TOKENS, OTHER_TOKEN,
    PAD_TOKEN,
)
from src.data.export import export_events
from src.data.sessions import build_sessions
from src.data.split import time_based_split
from src.data.vocab import build_vocab, encode_sequence

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

TensorFlow: 2.21.0
Project root: D:\Product Recommendation project


## Step 1 — Pull a small slice of data

We query **one day** of events (~600K rows) instead of the full 20M. Why: the model architecture doesn't care about data size, but our *iteration speed* while debugging does. If a bug takes 3 hours to surface because we ran it on the full dataset, that's 3 hours wasted per bug. On one day, the whole notebook runs in minutes, so we can fail fast and fix fast.

This reuses `src/data/export.py` unchanged — it's the same function the full pipeline will call, just with a narrower date range. That's the point of having it in `src/` rather than copy-pasted here.

In [2]:
SAMPLE_START_DATE = "2019-10-01"
SAMPLE_END_DATE = "2019-10-02"  # one day
sample_path = DATA_INTERIM_DIR / "events_sample.parquet"

export_events(start_date=SAMPLE_START_DATE, end_date=SAMPLE_END_DATE, output_path=sample_path)

events_df = pd.read_parquet(sample_path, columns=["user_session", "event_time", "product_id"])
print(f"{len(events_df):,} events loaded")
events_df.head()

  exported 200,000 rows so far...


  exported 400,000 rows so far...


  exported 600,000 rows so far...


  exported 800,000 rows so far...


  exported 1,000,000 rows so far...


  exported 1,200,000 rows so far...


  exported 1,244,245 rows so far...


1,244,245 events loaded


,user_session,event_time,product_id
0,72d76fde-8bb3-4e00-8c23-a032dfed738c,2019-10-01 00:00:00,44600062
1,9333dfbd-b87a-4708-9857-6336556b0fcc,2019-10-01 00:00:00,3900821
2,566511c2-e2e3-422b-b695-cf8e6e792ca8,2019-10-01 00:00:01,17200506
3,7c90fc70-0e80-4590-96f3-13c02c18c713,2019-10-01 00:00:01,1307067
4,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,2019-10-01 00:00:04,1004237


## Step 2 — Build sessions

A session is just: all events sharing a `user_session` id, sorted by time, collapsed to the ordered list of `product_id`s the user interacted with. `build_sessions()` (from `src/data/sessions.py`) also:

- **Drops 1-event sessions.** If a session only has one event, there's no "next item" to predict — nothing to train on.
- **Caps session length at the 95th percentile**, keeping the *most recent* items. A tiny number of extreme sessions (e.g. a bot, or someone who left 400 tabs open) would otherwise dominate how much padding every batch needs, wasting compute on nearly every training step for the sake of a handful of outlier sessions.

In [3]:
sessions_df = build_sessions(events_df)
print(f"{len(sessions_df):,} usable sessions")
sessions_df["session_len"].describe()

175,408 usable sessions


count    175408.000000
mean          5.927632
std           4.792284
min           2.000000
25%           2.000000
50%           4.000000
75%           8.000000
max          19.000000
Name: session_len, dtype: float64

## Step 3 — Vocabulary

There can be tens of thousands of distinct products even in one day. Asking the model's output layer to distinguish between all of them is expensive (more on this below, when we get to the softmax). So we keep only the top-N most frequent products as real tokens, and merge everything else into a single `<OTHER>` token — the model still learns *something* from long-tail products, it just can't tell two rare products apart from each other.

Token ids `0`, `1`, `2` are reserved: `<PAD>` (padding), `<UNK>` (a product never seen at all, only possible at inference), `<OTHER>` (seen in training, just not frequent enough for its own id). Real products start at id `3`, assigned **in descending frequency order** — id `3` is the single most-clicked product, id `4` the second most-clicked, and so on. That ordering isn't cosmetic — it matters later for how negative sampling works.

In [4]:
product_to_id = build_vocab(sessions_df)  # on this small sample, this may keep every distinct product
vocab_size = len(product_to_id) + NUM_SPECIAL_TOKENS
print(f"{len(product_to_id):,} real products in vocab -> {vocab_size:,} total token ids (incl. PAD/UNK/OTHER)")

40,000 real products in vocab -> 40,003 total token ids (incl. PAD/UNK/OTHER)


## Step 4 — Time-based train/val/test split

We split by *when a session started*, not randomly. If we shuffled sessions randomly into train/test, the model could end up training on a session from 2pm and being tested on a session from 10am the same day — effectively "seeing the future" relative to some of its test cases. That would inflate Recall@10/NDCG@10 in a way that wouldn't hold up once deployed, where the model genuinely never sees future data during training.

On this one-day sample the windows are tiny (hours, not days) — that's expected and fine for a prototype; the full run uses the `TEST_DAYS`/`VAL_DAYS` windows from `src/config.py`.

In [5]:
train_df, val_df, test_df = time_based_split(sessions_df, test_days=0.15, val_days=0.15)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

train=144,926  val=26,213  test=4,269


## Step 5 — From sessions to training pairs

The model doesn't train on whole sessions — it trains on **(what happened so far -> what happened next)** pairs. A session `[A, B, C, D]` becomes three training examples:

```
[A]       -> B
[A, B]    -> C
[A, B, C] -> D
```

This is why 2.8M usable sessions turn into ~15M training examples in the full dataset — one session generates (length - 1) examples.

**Padding:** every input sequence must be the same length to batch efficiently on a GPU/CPU (a batch is one big tensor — it can't have ragged rows). Short sequences get padded with `<PAD>` (id `0`) at the front (`padding='pre'`); sequences longer than `MAX_SEQ_LEN` get truncated from the front too (`truncating='pre'`), keeping the most recent items — the most relevant ones for predicting what's next.

In [6]:
def make_training_pairs(sessions_df, product_to_id):
    inputs, targets = [], []
    for seq in sessions_df["product_seq"]:
        encoded = encode_sequence(seq, product_to_id, default_token=OTHER_TOKEN)
        for i in range(1, len(encoded)):
            inputs.append(encoded[:i])
            targets.append(encoded[i])
    return inputs, targets


train_inputs, train_targets = make_training_pairs(train_df, product_to_id)
print(f"{len(train_inputs):,} training pairs from {len(train_df):,} sessions")

train_inputs_padded = pad_sequences(
    train_inputs, maxlen=MAX_SEQ_LEN, padding="pre", truncating="pre", value=PAD_TOKEN
)
train_targets = np.array(train_targets, dtype=np.int64)

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_inputs_padded, train_targets))
    .shuffle(buffer_size=len(train_inputs), seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
print(f"{len(train_inputs_padded) // BATCH_SIZE + 1} batches per epoch at batch size {BATCH_SIZE}")

694,256 training pairs from 144,926 sessions


2712 batches per epoch at batch size 256


## Concept: Embeddings

A product id like `4127` is just an arbitrary integer — the model shouldn't think product `4128` is "close to" product `4127` just because the numbers are adjacent. An **embedding layer** is a lookup table: `vocab_size` rows, each a learned vector of `EMBED_DIM` numbers. Product id `4127` gets mapped to a specific dense vector, and that vector is a *trainable parameter* — during training, the model adjusts it so that products which behave similarly (co-occur in sessions, get clicked/bought together) end up with similar vectors. This is strictly better than one-hot encoding, which would need a `vocab_size`-length vector per item and has no notion of similarity at all.

`mask_zero=True` tells the embedding layer that token id `0` (`<PAD>`) isn't a real item — it produces a *mask* alongside the output so downstream layers know which timesteps in a padded sequence to ignore.

## Concept: GRU (Gated Recurrent Unit)

A session is a *sequence* — order matters ("viewed a phone case after viewing a phone" is a different signal than the reverse). A GRU processes the sequence one item at a time, keeping a running **hidden state** that summarizes everything seen so far, updated at each step. After the last real item, that hidden state becomes our "context vector" — a compressed representation of the whole session so far, used to predict what comes next.

Why GRU specifically, not a plain RNN or an LSTM:
- **Plain RNN**: struggles with longer sequences — gradients tend to vanish or explode over many timesteps, so it effectively "forgets" early parts of the session.
- **LSTM**: solves that with gates, but has *more* gates/parameters (input, forget, output, cell state) — more compute per step, more to train.
- **GRU**: a simplified gating mechanism (reset + update gates, no separate cell state) that solves the same vanishing-gradient problem with fewer parameters — meaningfully cheaper per step, which matters directly for us since we're CPU-bound. In practice GRU and LSTM perform comparably on session-length sequences like these; GRU4Rec (the paper this project is named after) specifically found GRU sufficient for this exact problem.

## Concept: masking, explicitly

Because sequences are padded, some timesteps in every batch are "fake" (`<PAD>`). Without masking, the GRU would run its recurrence over those padding steps too, corrupting the hidden state with garbage from padding rather than real items. Keras layers can propagate a mask automatically in simple cases, but inside a custom model class it's easy for that to silently not happen — so below we compute the mask explicitly and pass it into the GRU rather than relying on implicit propagation.

## Concept: the softmax bottleneck, and why we use *sampled* softmax

The natural way to predict "what's next" is: take the context vector, project it to a score for *every* product in the vocabulary (`vocab_size` scores), and softmax over all of them. The problem: that projection is a `gru_units x vocab_size` matrix multiply, computed on **every single training step, for every example**. With a 40K vocabulary this dominates training cost so heavily that a full CPU run realistically takes 1-3 days (this is exactly what we worked out earlier) — nearly all of that time is spent scoring tens of thousands of products the model already knows are irrelevant to a given example.

**Sampled softmax** fixes this: instead of scoring all 40,000 products, score the *true* next item plus a small random sample of, say, 100 "negative" products, and only compute the softmax over those ~101. This is a well-established approximation (used in the original GRU4Rec paper, word2vec, and most large-vocabulary neural recommenders) — it gives a noisy but unbiased-in-expectation estimate of the true softmax gradient, at a small fraction of the cost. It only matters at *training* time — the negatives change every step, keeping the estimate honest; there's nothing to sample at inference time, since then we just want the real top-10.

One detail worth knowing: TensorFlow's `tf.nn.sampled_softmax_loss` uses a *log-uniform* sampler by default, which assumes class ids are roughly sorted by descending frequency — which is exactly how `vocab.py` assigns ids (id 3 = most frequent product). That's not a coincidence we have to work around; the vocab was built this way specifically so it lines up with how sampled softmax expects ids to be ordered.

In [7]:
class GRU4Rec(tf.keras.Model):
    """Embedding -> GRU -> (context vector). The output projection to
    vocab-sized logits is NOT a Dense layer here -- weights/bias are
    plain trainable variables so tf.nn.sampled_softmax_loss can use them
    directly during training, and full_logits() can use the same weights
    for exact (non-sampled) scoring at evaluation/inference time.
    """

    def __init__(self, vocab_size, embed_dim, gru_units):
        super().__init__()
        self.vocab_size = vocab_size
        self.gru_units = gru_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.gru = tf.keras.layers.GRU(gru_units)

    def build(self, input_shape):
        self.output_weights = self.add_weight(
            name="output_weights", shape=(self.vocab_size, self.gru_units),
            initializer="glorot_uniform",
        )
        self.output_bias = self.add_weight(
            name="output_bias", shape=(self.vocab_size,), initializer="zeros"
        )
        super().build(input_shape)

    def call(self, x, training=False):
        embedded = self.embedding(x)
        mask = self.embedding.compute_mask(x)
        return self.gru(embedded, mask=mask, training=training)  # (batch, gru_units) context vector

    def full_logits(self, context):
        """Exact scores over the full vocab -- only used for eval/inference,
        never inside the training loop."""
        return tf.matmul(context, self.output_weights, transpose_b=True) + self.output_bias


model = GRU4Rec(vocab_size=vocab_size, embed_dim=EMBED_DIM, gru_units=GRU_UNITS)
model.build(input_shape=(None, MAX_SEQ_LEN))
model.summary()

Model: "gru4_rec"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,160,387 (19.69 MB)

 Trainable params: 5,160,387 (19.69 MB)

 Non-trainable params: 0 (0.00 B)

## Concept: why a custom training loop instead of `model.fit(...)`

`model.fit()` expects a normal Keras loss shaped `loss(y_true, y_pred)`. Sampled softmax doesn't fit that shape — it needs direct access to the raw `output_weights`/`output_bias` matrices *and* the true labels *together*, so it can look up the true class's row and combine it with a handful of randomly sampled rows. That's not something you can express as a simple `(y_true, y_pred)` function.

So instead we write the training step manually with `tf.GradientTape`:
1. **Forward pass**: run the batch through the model to get context vectors.
2. **Loss**: `tf.nn.sampled_softmax_loss` compares the context vector against the true item's row in `output_weights` plus `NUM_SAMPLED_NEGATIVES` random rows — cheap, as discussed above.
3. **Gradient**: `tape.gradient(...)` computes how much each trainable variable (embedding table, GRU weights, output weights/bias) contributed to the loss.
4. **Update**: the optimizer nudges every variable a small step in the direction that reduces the loss.

`@tf.function` compiles this into a TensorFlow graph instead of running it step-by-step in plain Python, which matters a lot on CPU — graph execution avoids Python-interpreter overhead on every single batch.

In [8]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)


@tf.function
def train_step(batch_x, batch_y):
    with tf.GradientTape() as tape:
        context = model(batch_x, training=True)
        labels = tf.expand_dims(batch_y, axis=1)  # sampled_softmax_loss wants shape (batch, num_true=1)
        losses = tf.nn.sampled_softmax_loss(
            weights=model.output_weights,
            biases=model.output_bias,
            labels=labels,
            inputs=context,
            num_sampled=NUM_SAMPLED_NEGATIVES,
            num_classes=model.vocab_size,
        )
        loss = tf.reduce_mean(losses)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

In [9]:
EPOCHS_PROTOTYPE = 3  # deliberately small -- this is a smoke test, not a real training run

for epoch in range(EPOCHS_PROTOTYPE):
    start = time.time()
    batch_losses = []
    for batch_x, batch_y in train_ds:
        loss = train_step(batch_x, batch_y)
        batch_losses.append(float(loss))
    elapsed = time.time() - start
    print(f"epoch {epoch + 1}/{EPOCHS_PROTOTYPE}  loss={np.mean(batch_losses):.4f}  time={elapsed:.1f}s")

print("\nThis per-epoch time, scaled by (full training pairs / this sample's training pairs),")
print("gives a real measured estimate for the full 20M-row run -- far more trustworthy than")
print("the theoretical FLOP-based estimate discussed earlier.")

epoch 1/3  loss=3.6186  time=595.8s


epoch 2/3  loss=1.9120  time=508.9s


epoch 3/3  loss=1.3312  time=439.8s

This per-epoch time, scaled by (full training pairs / this sample's training pairs),
gives a real measured estimate for the full 20M-row run -- far more trustworthy than
the theoretical FLOP-based estimate discussed earlier.


## Concept: evaluation — Recall@10, NDCG@10, and a popularity baseline

Plain accuracy is the wrong metric here: this is a ranking problem (predict a top-10 list), not a single-label classification problem. We use:

- **Recall@10**: for each test example, was the true next item anywhere in the model's top-10 predictions? (1 if yes, 0 if no, averaged across examples.)
- **NDCG@10**: like Recall@10, but rewards ranking the correct item *higher* within the top 10 — a correct item at rank 1 scores more than one at rank 10.
- **Popularity baseline**: "always recommend the 10 globally most-clicked products," ignoring the session entirely. Because vocab ids are assigned in descending frequency order, the top-10 most popular ids are simply `NUM_SPECIAL_TOKENS .. NUM_SPECIAL_TOKENS + 9` — no extra bookkeeping needed.

The popularity baseline is the bar the model actually has to clear. A model that only matches it hasn't learned anything session-specific — it's just memorized what's generally popular, which needs zero machine learning to know.

In [10]:
def evaluate_model(model, eval_df, product_to_id, k=10, max_sessions=2000, seed=42):
    hits, ndcgs = [], []
    subset = eval_df.sample(n=min(max_sessions, len(eval_df)), random_state=seed)
    for seq in subset["product_seq"]:
        encoded = encode_sequence(seq, product_to_id, default_token=OTHER_TOKEN)
        if len(encoded) < 2:
            continue
        prefix, target = encoded[:-1], encoded[-1]
        padded = pad_sequences([prefix], maxlen=MAX_SEQ_LEN, padding="pre", truncating="pre", value=PAD_TOKEN)
        context = model(padded, training=False)
        logits = model.full_logits(context)[0].numpy()
        top_k = np.argsort(-logits)[:k]
        hit = target in top_k
        hits.append(1 if hit else 0)
        ndcgs.append(1 / np.log2(np.where(top_k == target)[0][0] + 2) if hit else 0.0)
    return np.mean(hits), np.mean(ndcgs)


def evaluate_popularity_baseline(eval_df, product_to_id, k=10, max_sessions=2000, seed=42):
    popular_top_k = set(range(NUM_SPECIAL_TOKENS, NUM_SPECIAL_TOKENS + k))
    hits = []
    subset = eval_df.sample(n=min(max_sessions, len(eval_df)), random_state=seed)
    for seq in subset["product_seq"]:
        encoded = encode_sequence(seq, product_to_id, default_token=OTHER_TOKEN)
        if len(encoded) < 2:
            continue
        target = encoded[-1]
        hits.append(1 if target in popular_top_k else 0)
    return np.mean(hits)

In [11]:
recall_10, ndcg_10 = evaluate_model(model, val_df, product_to_id)
popularity_recall_10 = evaluate_popularity_baseline(val_df, product_to_id)

print(f"GRU4Rec (prototype, {EPOCHS_PROTOTYPE} epochs, 1 day of data):")
print(f"  Recall@10 = {recall_10:.4f}")
print(f"  NDCG@10   = {ndcg_10:.4f}")
print(f"Popularity baseline:")
print(f"  Recall@10 = {popularity_recall_10:.4f}")
print("\nOn this little data/epochs the model may not clearly beat the baseline yet --")
print("that's expected and fine. The point of this run is verifying the pipeline works")
print("and measuring epoch time, not producing a good model.")

GRU4Rec (prototype, 3 epochs, 1 day of data):
  Recall@10 = 0.3915
  NDCG@10   = 0.2811
Popularity baseline:
  Recall@10 = 0.0700

On this little data/epochs the model may not clearly beat the baseline yet --
that's expected and fine. The point of this run is verifying the pipeline works
and measuring epoch time, not producing a good model.


## What changes when we scale this up to the full 20M rows

Nothing conceptually — every piece above (sessions, vocab, pairs, model, sampled softmax, evaluation) is exactly what runs at full scale. What actually changes:

- `export_events()` called with the full date range instead of one day.
- `VOCAB_SIZE` in `src/config.py` (40,000) instead of "however many products exist in one day."
- `EPOCHS` (10, from config) instead of `EPOCHS_PROTOTYPE` (3).
- The measured per-epoch time above, scaled by the ratio of training-pair counts, gives a real estimate for the full run.

**Next steps:**
1. Promote this logic into `src/model/gru4rec.py` (model class) and `src/model/train.py` (training loop + checkpointing), so it's runnable as a script, not just a notebook.
2. `src/model/evaluate.py` gets the `evaluate_model`/`evaluate_popularity_baseline` logic above.
3. Run the full pipeline (`export -> sessions -> vocab -> split -> train`) against the full 20M rows. If the measured epoch time above makes that impractical on this CPU, that's the point to move the actual training run to a free-tier GPU notebook (Colab/Kaggle) — the exported Parquet + these same modules travel with it unchanged.

## Recap: how this supports the retrain strategy

The retrain plan discussed earlier (sliding-window full retrain + frequent warm-start fine-tune) depends on being able to **resume training from a saved checkpoint** rather than always starting from random weights. That's exactly what `model.save_weights(...)` / `model.load_weights(...)` give us here — the pattern below is what `train.py` will do for real, just illustrated on the prototype model.

In [12]:
checkpoint_path = PROJECT_ROOT / "models" / "gru4rec_prototype.weights.h5"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
model.save_weights(checkpoint_path)
print(f"Saved prototype weights -> {checkpoint_path}")
print("A warm-start fine-tune would call model.load_weights(checkpoint_path) here instead of")
print("model.build(...) + random init, then continue training only on newly accumulated data.")

Saved prototype weights -> D:\Product Recommendation project\models\gru4rec_prototype.weights.h5
A warm-start fine-tune would call model.load_weights(checkpoint_path) here instead of
model.build(...) + random init, then continue training only on newly accumulated data.
